# 07. 월간 리포트 이메일 발송

이 Notebook에서 하는 일:
1. 이번 달 포트폴리오 데이터 수집
2. HTML 이메일 리포트 생성
3. Gmail SMTP로 이메일 발송
4. 발송 이력 DB 저장

In [1]:
import pandas as pd
import sqlite3, os, smtplib
from pathlib import Path
from dotenv import load_dotenv
from datetime import date
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import yaml

load_dotenv()
PROJECT_ROOT = Path.cwd()

with open(PROJECT_ROOT / 'config.yaml', encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

MONTHLY_EXPENSE = CONFIG['user']['monthly_expense']
TARGET          = CONFIG['portfolio']
USER_NAME       = CONFIG['user']['name']
EMAIL_TO        = CONFIG['alert']['email']
inflation_rate  = CONFIG.get('inflation', {}).get('assumed_rate', 0.025)
db_path         = PROJECT_ROOT / 'portfolio.db'

GMAIL_ADDRESS  = os.getenv('GMAIL_ADDRESS', '')
GMAIL_PASSWORD = os.getenv('GMAIL_APP_PASSWORD', '')

# 국민연금
PENSION_CFG   = CONFIG.get('income', {}).get('national_pension', {})
base_pension  = PENSION_CFG.get('base_amount', 0)
pension_start = PENSION_CFG.get('start_date', None)
today_date    = date.today()

if pension_start:
    py, pm = map(int, pension_start.split('-'))
    if today_date >= date(py, pm, 1):
        years_since = (today_date.year - py) + (today_date.month - pm) / 12
        pension_income = base_pension * (1 + inflation_rate) ** years_since
        months_to_pension = 0
    else:
        pension_income = 0
        months_to_pension = (py - today_date.year) * 12 + (pm - today_date.month)
else:
    pension_income = 0
    months_to_pension = 0

print(f'수신 이메일: {EMAIL_TO}')
print(f'발신 계정:   {GMAIL_ADDRESS}')
print(f'앱 비밀번호: {"설정됨" if GMAIL_PASSWORD else "❌ 미설정"}')

수신 이메일: jonghun.kim84@gmail.com
발신 계정:   jonghun.kim84@gmail.com
앱 비밀번호: 설정됨


In [2]:
# ── 데이터 수집 ──────────────────────────────────────────────
assets_path = PROJECT_ROOT / 'assets.csv'
sample_path = PROJECT_ROOT / 'assets_sample.csv'
df = pd.read_csv(assets_path if assets_path.exists() else sample_path, encoding='utf-8-sig')

mask = df['current_value'].isna() | (df['current_value'] == 0)
df.loc[mask, 'current_value'] = df.loc[mask, 'quantity'] * df.loc[mask, 'unit_price']
df['current_value'] = pd.to_numeric(df['current_value'], errors='coerce').fillna(0)


# ── 비활성(만료) 자산 제외 ──────────────────────────────────
if 'is_active' not in df.columns: df['is_active'] = 1
if 'maturity_date' not in df.columns: df['maturity_date'] = ''
df['is_active'] = df['is_active'].fillna(1).astype(int)
inactive_n = int((df['is_active'] == 0).sum())
df = df[df['is_active'] == 1].reset_index(drop=True)
if inactive_n > 0:
    print(f'ℹ️  비활성(만료) 자산 {inactive_n}개 제외 → 활성 {len(df)}개 사용')
BUCKET_MAP = {'cash': 1, 'bond': 2, 'tdf': 2, 'fund': 2, 'equity': 3, 'income': 3}
df['bucket'] = df['asset_type'].str.lower().map(BUCKET_MAP).fillna(0).astype(int)

bucket_sum = df.groupby('bucket')['current_value'].sum()
type_sum   = df.groupby('asset_type')['current_value'].sum()
total      = df['current_value'].sum()
b1 = bucket_sum.get(1, 0)
b2 = bucket_sum.get(2, 0)
b3 = bucket_sum.get(3, 0)
months_covered = b1 / MONTHLY_EXPENSE

# DB에서 최신 데이터
conn = sqlite3.connect(db_path)
cur  = conn.cursor()

cur.execute('SELECT total_score, cash_score, seq_score, conc_score, level FROM risk_scores ORDER BY date DESC LIMIT 1')
risk_row    = cur.fetchone()
total_score = risk_row[0] if risk_row else 0
cash_score  = risk_row[1] if risk_row else 0
seq_score   = risk_row[2] if risk_row else 0
conc_score  = risk_row[3] if risk_row else 0
risk_level  = risk_row[4] if risk_row else 'green'

cur.execute("SELECT message FROM recommendations WHERE rule_id='ai_summary' ORDER BY date DESC LIMIT 1")
ai_row     = cur.fetchone()
ai_summary = ai_row[0] if ai_row else '요약 없음. 05_llm_summary.ipynb를 먼저 실행하세요.'

# 인출 데이터 — actual_amount 우선
cur.execute('SELECT amount, actual_amount, guardrail_applied FROM withdrawal_log ORDER BY date DESC LIMIT 1')
wd_row = cur.fetchone()
if wd_row:
    recommended_wd    = wd_row[0] or 0
    actual_wd         = wd_row[1]   # None이면 미입력
    guardrail_applied = wd_row[2] or 0
    display_wd        = actual_wd if actual_wd is not None else recommended_wd
    wd_label          = '실제 인출액' if actual_wd is not None else '권장 인출액'
else:
    recommended_wd    = max(0, MONTHLY_EXPENSE - pension_income)
    actual_wd         = None
    display_wd        = recommended_wd
    wd_label          = '권장 인출액'
    guardrail_applied = 0

# 리밸런싱 항목
cur.execute("SELECT message FROM recommendations WHERE rule_id LIKE 'rebalance_%' AND date=date('now') ORDER BY date DESC")
rebalance_items = [row[0] for row in cur.fetchall()]
conn.close()

today_str   = date.today().strftime('%Y년 %m월 %d일')
month_str   = date.today().strftime('%Y년 %m월')
level_map   = {'green': '🟢 녹색(안전)', 'yellow': '🟡 황색(주의)', 'red': '🔴 적색(위험)'}
level_label = level_map.get(risk_level, '확인 필요')
net_wd      = max(0, MONTHLY_EXPENSE - pension_income)

print(f'총 자산: {total:,.0f}원')
print(f'위험 등급: {level_label}  ({total_score:.1f}점)')
print(f'현금 {months_covered:.1f}개월치  |  {wd_label}: {display_wd:,.0f}원')


총 자산: 804,317,346원
위험 등급: 🟡 황색(주의)  (30.0점)
현금 67.0개월치  |  실제 인출액: 4,500,000원


## 1. HTML 이메일 리포트 생성

In [3]:
def build_html_report():
    level_color = {'green': '#00B050', 'yellow': '#FFC000', 'red': '#FF0000'}.get(risk_level, '#00B050')
    level_bg    = {'green': '#E2EFDA', 'yellow': '#FFF2CC', 'red': '#FCE4D6'}.get(risk_level, '#E2EFDA')
    cash_status = '✅ 충분' if months_covered >= 12 else ('⚠️ 주의' if months_covered >= 6 else '🚨 부족')
    guard_txt   = ' (가드레일 하향 적용)' if guardrail_applied else ''

    # 보유 자산 행
    asset_rows = ''
    for _, row in df.iterrows():
        bucket_names = {1: '현금성', 2: '채권/TDF', 3: '성장/인컴'}
        bucket_label = bucket_names.get(row['bucket'], '-')
        asset_rows += f"""
        <tr>
          <td style="padding:7px 10px;border-bottom:1px solid #f0f0f0;">{row['account_name']}</td>
          <td style="padding:7px 10px;border-bottom:1px solid #f0f0f0;">{row['asset_name']}</td>
          <td style="padding:7px 10px;border-bottom:1px solid #f0f0f0;text-align:center;">{bucket_label}</td>
          <td style="padding:7px 10px;border-bottom:1px solid #f0f0f0;text-align:right;">{row['current_value']:,.0f}원</td>
        </tr>"""

    # 리밸런싱
    if rebalance_items:
        rebalance_html = '<ul style="margin:8px 0;padding-left:20px;">'
        for item in rebalance_items:
            rebalance_html += f'<li style="margin:4px 0;color:#C00000;">{item}</li>'
        rebalance_html += '</ul>'
    else:
        rebalance_html = '<p style="color:#00B050;margin:0;">✅ 모든 자산군이 목표 비중 범위 내에 있습니다.</p>'

    # AI 요약
    ai_lines = ''.join([
        f'<p style="margin:5px 0;">{i+1}. {line}</p>'
        for i, line in enumerate(ai_summary.split('\n')) if line.strip()
    ])

    # 수입·지출 섹션
    if pension_income > 0:
        income_html = f"""
      <p style="margin:4px 0;">월 생활비: {MONTHLY_EXPENSE:,.0f}원</p>
      <p style="margin:4px 0;">국민연금 수령: <b style="color:#27ae60;">{pension_income:,.0f}원</b></p>
      <p style="margin:4px 0;">포트폴리오 {wd_label}: <b>{display_wd:,.0f}원</b>{guard_txt}</p>
      <p style="margin:4px 0;">연간 인출률: {net_wd*12/total*100:.1f}% ({'✅ 4% 이내' if net_wd*12/total <= 0.04 else '⚠️ 4% 초과'})</p>"""
    elif months_to_pension > 0:
        income_html = f"""
      <p style="margin:4px 0;">월 생활비: {MONTHLY_EXPENSE:,.0f}원</p>
      <p style="margin:4px 0;">국민연금: 미수령 ({months_to_pension}개월 후 {base_pension:,.0f}원 개시 예정)</p>
      <p style="margin:4px 0;">포트폴리오 {wd_label}: <b>{display_wd:,.0f}원</b>{guard_txt}</p>
      <p style="margin:4px 0;">연간 인출률: {net_wd*12/total*100:.1f}% — 연금 개시 후 개선 예정</p>"""
    else:
        income_html = f"""
      <p style="margin:4px 0;">월 생활비: {MONTHLY_EXPENSE:,.0f}원</p>
      <p style="margin:4px 0;">포트폴리오 {wd_label}: <b>{display_wd:,.0f}원</b>{guard_txt}</p>"""

    html = f"""
<!DOCTYPE html>
<html lang="ko">
<head><meta charset="UTF-8"><title>{month_str} 포트폴리오 월간 리포트</title></head>
<body style="font-family:'맑은 고딕',Arial,sans-serif;background:#f5f5f5;margin:0;padding:20px;">
<div style="max-width:680px;margin:0 auto;background:white;border-radius:12px;
            box-shadow:0 2px 8px rgba(0,0,0,.1);overflow:hidden;">

  <div style="background:linear-gradient(135deg,#1F3864,#2E75B6);padding:28px 32px;color:white;">
    <h1 style="margin:0;font-size:20px;">🏦 은퇴 포트폴리오 월간 리포트</h1>
    <p style="margin:6px 0 0;opacity:.85;font-size:13px;">{USER_NAME}님 · {today_str} 기준</p>
  </div>

  <div style="padding:24px 32px;">

    <table style="width:100%;border-collapse:collapse;margin-bottom:24px;">
      <tr>
        <td style="width:50%;padding-right:8px;">
          <div style="background:#EBF3FB;border-radius:8px;padding:14px 16px;">
            <div style="font-size:11px;color:#666;">총 자산</div>
            <div style="font-size:20px;font-weight:700;color:#1F3864;">{total/1e8:.2f}억원</div>
            <div style="font-size:11px;color:#888;">월 생활비 {MONTHLY_EXPENSE/10000:.0f}만원</div>
          </div>
        </td>
        <td style="width:50%;padding-left:8px;">
          <div style="background:{level_bg};border-radius:8px;padding:14px 16px;border-left:4px solid {level_color};">
            <div style="font-size:11px;color:#666;">위험 등급</div>
            <div style="font-size:20px;font-weight:700;color:{level_color};">{level_label}</div>
            <div style="font-size:11px;color:#888;">종합 점수 {total_score:.1f}점 / 100점</div>
          </div>
        </td>
      </tr>
    </table>

    <h2 style="font-size:15px;color:#1F3864;border-bottom:2px solid #1F3864;padding-bottom:6px;margin-bottom:14px;">📦 버킷별 자산 현황</h2>
    <table style="width:100%;border-collapse:collapse;margin-bottom:24px;font-size:13px;">
      <tr style="background:#1F3864;color:white;">
        <th style="padding:8px 12px;text-align:left;">버킷</th>
        <th style="padding:8px 12px;text-align:right;">금액</th>
        <th style="padding:8px 12px;text-align:center;">비중</th>
        <th style="padding:8px 12px;text-align:center;">생활비</th>
        <th style="padding:8px 12px;text-align:center;">상태</th>
      </tr>
      <tr style="background:#f8f9fa;">
        <td style="padding:9px 12px;">버킷 1 (현금성)</td>
        <td style="padding:9px 12px;text-align:right;">{b1:,.0f}원</td>
        <td style="padding:9px 12px;text-align:center;">{b1/total*100:.1f}%</td>
        <td style="padding:9px 12px;text-align:center;">{months_covered:.1f}개월</td>
        <td style="padding:9px 12px;text-align:center;">{cash_status}</td>
      </tr>
      <tr>
        <td style="padding:9px 12px;">버킷 2 (채권/TDF/펀드)</td>
        <td style="padding:9px 12px;text-align:right;">{b2:,.0f}원</td>
        <td style="padding:9px 12px;text-align:center;">{b2/total*100:.1f}%</td>
        <td style="padding:9px 12px;text-align:center;">{b2/MONTHLY_EXPENSE:.1f}개월</td>
        <td style="padding:9px 12px;text-align:center;">-</td>
      </tr>
      <tr style="background:#f8f9fa;">
        <td style="padding:9px 12px;">버킷 3 (성장/인컴)</td>
        <td style="padding:9px 12px;text-align:right;">{b3:,.0f}원</td>
        <td style="padding:9px 12px;text-align:center;">{b3/total*100:.1f}%</td>
        <td style="padding:9px 12px;text-align:center;">{b3/MONTHLY_EXPENSE:.1f}개월</td>
        <td style="padding:9px 12px;text-align:center;">-</td>
      </tr>
    </table>

    <h2 style="font-size:15px;color:#1F3864;border-bottom:2px solid #1F3864;padding-bottom:6px;margin-bottom:14px;">🔍 위험 점수 상세</h2>
    <table style="width:100%;border-collapse:collapse;margin-bottom:24px;font-size:13px;">
      <tr style="background:#1F3864;color:white;">
        <th style="padding:8px 12px;text-align:left;">항목</th>
        <th style="padding:8px 12px;text-align:center;">점수</th>
        <th style="padding:8px 12px;text-align:center;">가중치</th>
      </tr>
      <tr style="background:#f8f9fa;"><td style="padding:8px 12px;">현금 위험</td><td style="padding:8px 12px;text-align:center;">{cash_score:.0f}점</td><td style="padding:8px 12px;text-align:center;">40%</td></tr>
      <tr><td style="padding:8px 12px;">순서 위험</td><td style="padding:8px 12px;text-align:center;">{seq_score:.0f}점</td><td style="padding:8px 12px;text-align:center;">40%</td></tr>
      <tr style="background:#f8f9fa;"><td style="padding:8px 12px;">집중 위험</td><td style="padding:8px 12px;text-align:center;">{conc_score:.0f}점</td><td style="padding:8px 12px;text-align:center;">20%</td></tr>
      <tr style="background:{level_bg};font-weight:700;"><td style="padding:8px 12px;">종합</td><td style="padding:8px 12px;text-align:center;color:{level_color};">{total_score:.1f}점</td><td style="padding:8px 12px;text-align:center;color:{level_color};">{level_label}</td></tr>
    </table>

    <h2 style="font-size:15px;color:#1F3864;border-bottom:2px solid #1F3864;padding-bottom:6px;margin-bottom:14px;">💸 이번 달 수입·지출</h2>
    <div style="background:#f8f9fa;border-radius:8px;padding:14px 18px;margin-bottom:24px;font-size:13px;">
      {income_html}
    </div>

    <h2 style="font-size:15px;color:#1F3864;border-bottom:2px solid #1F3864;padding-bottom:6px;margin-bottom:14px;">⚖️ 리밸런싱 현황</h2>
    <div style="margin-bottom:24px;font-size:13px;">{rebalance_html}</div>

    <h2 style="font-size:15px;color:#1F3864;border-bottom:2px solid #1F3864;padding-bottom:6px;margin-bottom:14px;">🤖 AI 포트폴리오 요약</h2>
    <div style="background:#EBF3FB;border-left:4px solid #2E75B6;border-radius:6px;
                padding:14px 18px;margin-bottom:24px;font-size:13px;line-height:1.7;">
      {ai_lines}
    </div>

    <h2 style="font-size:15px;color:#1F3864;border-bottom:2px solid #1F3864;padding-bottom:6px;margin-bottom:14px;">📋 보유 자산 상세</h2>
    <table style="width:100%;border-collapse:collapse;font-size:12px;margin-bottom:24px;">
      <tr style="background:#1F3864;color:white;">
        <th style="padding:7px 10px;text-align:left;">계좌</th>
        <th style="padding:7px 10px;text-align:left;">자산명</th>
        <th style="padding:7px 10px;text-align:center;">버킷</th>
        <th style="padding:7px 10px;text-align:right;">평가액</th>
      </tr>
      {asset_rows}
      <tr style="background:#1F3864;color:white;font-weight:700;">
        <td colspan="3" style="padding:8px 10px;">합계</td>
        <td style="padding:8px 10px;text-align:right;">{total:,.0f}원</td>
      </tr>
    </table>

  </div>

  <div style="background:#f1f3f5;padding:16px 32px;font-size:11px;color:#888;">
    이 리포트는 은퇴 포트폴리오 AI 시스템이 자동 생성했습니다. · {today_str}
  </div>

</div>
</body></html>
"""
    return html


report_html = build_html_report()
print(f'HTML 리포트 생성 완료 ({len(report_html):,} bytes)')

HTML 리포트 생성 완료 (18,945 bytes)


## 2. 리포트 미리보기

In [4]:
from IPython.display import display, HTML
display(HTML(report_html))

## 3. Gmail로 이메일 발송

In [5]:
def send_email(subject, html_body, to_email, from_email, app_password):
    if not app_password:
        print('❌ Gmail 앱 비밀번호가 설정되지 않았습니다.')
        return False

    from email.header import Header
    msg = MIMEMultipart('alternative')
    msg['Subject'] = Header(subject, 'utf-8')
    msg['From']    = from_email
    msg['To']      = to_email
    msg.attach(MIMEText(html_body, 'html', 'utf-8'))

    try:
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
            server.login(from_email, app_password)
            server.send_message(msg)
        print(f'✅ 이메일 발송 성공!')
        print(f'   수신: {to_email}')
        print(f'   제목: {subject}')
        return True
    except smtplib.SMTPAuthenticationError:
        print('❌ 인증 실패: Gmail 앱 비밀번호를 확인하세요.')
        return False
    except Exception as e:
        print(f'❌ 발송 실패: {e}')
        return False


subject = f'[포트폴리오 리포트] {month_str} 월간 현황 — {level_label}'

print(f'이메일 발송 중...')
print(f'제목: {subject}')
print()

success = send_email(
    subject      = subject,
    html_body    = report_html,
    to_email     = EMAIL_TO,
    from_email   = GMAIL_ADDRESS,
    app_password = GMAIL_PASSWORD
)

이메일 발송 중...
제목: [포트폴리오 리포트] 2026년 05월 월간 현황 — 🟡 황색(주의)

✅ 이메일 발송 성공!
   수신: jonghun.kim84@gmail.com
   제목: [포트폴리오 리포트] 2026년 05월 월간 현황 — 🟡 황색(주의)


## 4. 발송 이력 DB 저장

In [6]:
today = str(date.today())
conn  = sqlite3.connect(db_path)
cur   = conn.cursor()

cur.execute('''
    INSERT INTO alert_log (date, channel, subject, sent_ok)
    VALUES (?, ?, ?, ?)
''', (today, 'email', subject, 1 if success else 0))

conn.commit()
conn.close()

print(f'발송 이력 저장 완료 ({today})')
print(f'결과: {"성공" if success else "실패"}')

발송 이력 저장 완료 (2026-05-21)
결과: 성공


## 5. 발송 이력 조회

In [7]:
conn = sqlite3.connect(db_path)
history = pd.read_sql_query(
    'SELECT date, channel, subject, sent_ok FROM alert_log ORDER BY date DESC LIMIT 12', conn)
conn.close()

if history.empty:
    print('발송 이력이 없습니다.')
else:
    history['sent_ok'] = history['sent_ok'].map({1: '✅ 성공', 0: '❌ 실패'})
    history.columns = ['날짜', '채널', '제목', '결과']
    print('=== 최근 이메일 발송 이력 ===')
    print(history.to_string(index=False))

=== 최근 이메일 발송 이력 ===
        날짜    채널                                     제목   결과
2026-05-21 email [포트폴리오 리포트] 2026년 05월 월간 현황 — 🟡 황색(주의) ✅ 성공
2026-05-20 email [포트폴리오 리포트] 2026년 05월 월간 현황 — 🟡 황색(주의) ✅ 성공
2026-05-20 email [포트폴리오 리포트] 2026년 05월 월간 현황 — 🟡 황색(주의) ✅ 성공
